In [ ]:
CONFIG = {
    "input_file" : "../../DATASET/v3test_queries_final.json",
    "output_dir" : ".",           # folder simpan hasil JSON
    "alpha"      : 0.50,          #dinamis
    "opensearch" : {
        "host"    : "localhost",
        "port"    : 9200,
        "user"    : "admin",
        "password": "KoTA404TABAH!"
    },
    "index_name" : "books2",
    "sbert_model": "all-MiniLM-L6-v2"
}

In [ ]:
import copy, json, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, HTML
from collections import defaultdict

from opensearchpy import OpenSearch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

In [ ]:
cfg = CONFIG['opensearch']
client = OpenSearch(
    hosts=[{'host': cfg['host'], 'port': cfg['port']}],
    http_auth=(cfg['user'], cfg['password']),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
    timeout=60
)
info = client.info()
print(f"✓ OpenSearch connected — version: {info['version']['number']}")

In [ ]:
sbert_model = SentenceTransformer('sbert_model')
print('done')

In [ ]:
def get_candidates(query: str) -> list[dict]:
    body = {
        "suggest": {
            "suggest_by_title": {
                "prefix": query,
                "completion": {
                    "field": "suggest_title",
                    "size": 10,
                    "skip_duplicates": True
                }
            },
            "suggest_by_author": {
                "prefix": query,
                "completion": {
                    "field": "suggest_author",
                    "size": 10,
                    "skip_duplicates": True
                }
            }
        },
        "query": {
            "bool": {
                "should": [
                    {"match": {"title":  {"query": query, "fuzziness": "2"}}},
                    {"match": {"author": {"query": query, "fuzziness": "2"}}}
                ],
                "minimum_should_match": "1"
            }
        },
        "size": 10,
        "_source": ["title", "author", "description", "numRatings", "pop_weight"]
    }

    response = client.search(index=INDEX_NAME, body=body)
    seen     = set()
    result   = []

    # Completion suggester (prioritas utama)
    for key in ["suggest_by_title", "suggest_by_author"]:
        for opt in response["suggest"][key][0]["options"]:
            if len(result) >= 10:
                break
            src   = opt["_source"]
            title = src.get("title", "")
            if not title or title in seen:
                continue
            seen.add(title)
            result.append({
                "title":       title,
                "author":      src.get("author", ""),
                "description": src.get("description", ""),
                "numRatings":  int(src.get("numRatings", 0)),
                "pop_weight":  float(src.get("pop_weight", 0.0)),
                "source":      "completion"
            })

    # Fuzzy match fallback jika completion < 10
    if len(result) < 10:
        for hit in response.get("hits", {}).get("hits", []):
            if len(result) >= 10:
                break
            src   = hit["_source"]
            title = src.get("title", "")
            if not title or title in seen:
                continue
            seen.add(title)
            result.append({
                "title":       title,
                "author":      src.get("author", ""),
                "description": src.get("description", ""),
                "numRatings":  int(src.get("numRatings", 0)),
                "pop_weight":  float(src.get("pop_weight", 0.0)),
                "source":      "fuzzy"
            })

    return result

print('done')

In [ ]:
def normalize(scores):
    min_s, max_s = min(scores), max(scores)
    if max_s - min_s == 0:
        return [0.0] * len(scores)
    return [(s - min_s) / (max_s - min_s) for s in scores]

def rerank_bm25(query, candidates):
    corpus           = [f"{c['title']} {c['author']}" for c in candidates]
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    #tokenized_query  = query.lower().split()
    bm25             = BM25Okapi(tokenized_corpus)
    scores   = bm25.get_scores(query.lower().split())
    for i, c in enumerate(candidates):
        c["score_bm25"] = float(scores[i])
    return sorted(candidates, key=lambda x: x["score_bm25"], reverse=True)

def rerank_sbert(query, candidates):
    # model         = get_sbert_model()
    corpus        = [f"{c['title']} {c['author']} {c['description']}" for c in candidates]
    query_vec     = sbert_model.encode([query])
    candidate_vec = sbert_model.encode(corpus)
    scores = cosine_similarity(query_vec, candidate_vec)[0]
    for i, c in enumerate(candidates):
        c["score_sbert"] = float(scores[i])
    return sorted(candidates, key=lambda x: x["score_sbert"], reverse=True)

def rerank_hybrid(query, candidates, alpha):
    corpus     = [f"{c['title']} {c['author']}" for c in candidates]
    bm25       = BM25Okapi([d.lower().split() for d in corpus])
    bm25_raw   = bm25.get_scores(query.lower().split())
    sbert_corp = [f"{c['title']} {c['author']} {c['description']}" for c in candidates]
    q_vec      = sbert_model.encode([query])
    c_vec      = sbert_model.encode(sbert_corp)
    sbert_raw  = cosine_similarity(q_vec, c_vec)[0]
    bm25_norm  = normalize(list(bm25_raw))
    sbert_norm = normalize([float(s) for s in sbert_raw])
    for i, c in enumerate(candidates):
        c['score_bm25']   = float(bm25_raw[i])
        c['score_sbert']  = float(sbert_raw[i])
        c['score_hybrid'] = float((1 - alpha) * bm25_norm[i] + alpha * sbert_norm[i])
    return sorted(candidates, key=lambda x: x['score_hybrid'], reverse=True)

print('done')

In [ ]:
alpha = CONFIG['alpha']

with open(CONFIG['input_file'], 'r', encoding='utf-8') as f:
    queries = json.load(f)

print(f'Total query: {len(queries)} | alpha={alpha}')
print('Memproses...')

results = []
for q in tqdm(queries, desc='Eksperimen'):
    qtext      = q['query_text']
    candidates = get_candidates(qtext)

    if not candidates:
        results.append({**q, 'alpha': alpha, 'status': 'FAILED',
                        'candidates': {'completion':[],'bm25':[],'sbert':[],'hybrid':[]}})
        continue

    comp   = copy.deepcopy(candidates)
    bm25_r = rerank_bm25(qtext,   copy.deepcopy(candidates))
    sbert_r= rerank_sbert(qtext,  copy.deepcopy(candidates))
    hyb_r  = rerank_hybrid(qtext, copy.deepcopy(candidates), alpha=alpha)

    # Hapus description dari hasil
    for lst in [comp, bm25_r, sbert_r, hyb_r]:
        for c in lst: c.pop('description', None)

    results.append({
        'query_id'  : q['query_id'],
        'query_text': qtext,
        'query_type': q['query_type'],
        'title'     : q['title'],
        'author'    : q['author'],
        'alpha'     : alpha,
        'status'    : 'OK',
        'candidates': {'completion': comp, 'bm25': bm25_r, 'sbert': sbert_r, 'hybrid': hyb_r}
    })

# Simpan ke JSON
alpha_str   = f"{alpha:.2f}".replace('.', '')
output_file = os.path.join(CONFIG['output_dir'], f'v5experiment_results_alpha{alpha_str}.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

success = sum(1 for r in results if r['status'] == 'OK')
print(f'\n✓ Selesai! Berhasil: {success}/{len(results)} → {output_file}')